# 🧠 PyTorch 딥러닝 실습 — MNIST 손글씨 분류

---

### 실습 목표
- PyTorch로 간단한 신경망을 만들어 본다
- 손글씨 숫자(0~9) 이미지를 자동으로 분류하는 AI를 학습시킨다
- 순전파 → 손실 계산 → 역전파 → 업데이트 사이클을 직접 체험한다

### 사전 준비
1. **GPU 켜기**: 상단 메뉴 → `런타임` → `런타임 유형 변경` → 하드웨어 가속기를 **T4 GPU**로 선택 → 저장
2. 각 셀을 **위에서 아래로 순서대로** 실행하세요 (▶️ 버튼 또는 `Shift + Enter`)

---
## ⚡ Step 0. GPU 연결 확인

In [ ]:
# ============================================================
# ⚡ GPU 연결 확인
# ============================================================
import torch

print(f"GPU 사용 가능: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    print("✅ GPU가 정상적으로 연결되었습니다!")
else:
    print("⚠️ GPU가 연결되지 않았습니다!")
    print("  → 상단 메뉴 > 런타임 > 런타임 유형 변경 > T4 GPU 선택 후 저장해 주세요")
    print("  → CPU로도 실행은 가능하지만, 학습이 좀 더 느릴 수 있습니다")

---
## 📦 Step 1. 라이브러리 불러오기

In [ ]:
# ============================================================
# 📦 필요한 도구들 불러오기
# ============================================================
# Colab에는 PyTorch가 이미 설치되어 있어서
# 별도 설치(pip install) 없이 바로 import 하면 된다!
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

print("✅ 모든 라이브러리를 성공적으로 불러왔습니다!")

---
## 📂 Step 2. MNIST 데이터 불러오기

MNIST = 0~9까지 **손으로 쓴 숫자** 이미지 7만 장
- 학습용 60,000장 (교과서)
- 테스트용 10,000장 (시험지)
- 각 이미지는 28×28 픽셀 흑백

In [ ]:
# ============================================================
# 📂 MNIST 데이터셋 준비하기
# ============================================================

transform = transforms.Compose([
    # ① ToTensor(): 이미지를 PyTorch 텐서로 변환 => 흑백 이미지 (0,1)
    transforms.ToTensor(),

    # ② Normalize(): 데이터의 분포를 조정
    #    값의 범위가 0.0~1.0 → -1.0~1.0으로 바뀜
    transforms.Normalize((0.5,), (0.5,))
])

# 학습용 데이터셋 (60,000장)
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# 테스트용 데이터셋 (10,000장)
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# DataLoader: 데이터를 64장씩 배치로 나눠서 전달
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print(f"✅ 학습 데이터: {len(train_dataset)}장")
print(f"✅ 테스트 데이터: {len(test_dataset)}장")
print(f"   이미지 크기: 28 × 28 픽셀 (흑백)")
print(f"   배치 사이즈: 64장씩")
print(f"   학습 배치 수: {len(train_loader)}개 (= 60,000 ÷ 64)")

---
## 👀 Step 3. 데이터 미리보기

실제로 MNIST 데이터가 어떻게 생겼는지 눈으로 확인해보자!

In [ ]:
# ============================================================
# 👀 MNIST 데이터 미리보기
# ============================================================

fig, axes = plt.subplots(1, 10, figsize=(15, 2))

for i in range(10):
    image, label = train_dataset[i]
    axes[i].imshow(image.squeeze(), cmap='gray')
    axes[i].set_title(f"정답: {label}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("\n💡 위 이미지들이 28×28 픽셀 흑백 손글씨 숫자입니다!")
print("   이런 이미지를 보고 0~9 중 어떤 숫자인지 맞추는 AI를 만들 거예요.")

---
## 🏗️ Step 4. 신경망 모델 만들기

> ✨ **CNN(합성곱 신경망)으로 업그레이드!** `nn.Conv2d` 사용으로 정확도 ~97% → ~99% 향상

| 구분 | 기존 MLP | CNN (현재) |
|------|----------|-----------|
| 입력 처리 | 784개 숫자로 펼쳐서 처리 | 28×28 이미지 그대로 처리 |
| 공간 정보 | ❌ 위치 정보 버림 | ✅ 이웃 픽셀 관계 유지 |
| 예상 정확도 | ~97% | **~99%** |

구조:
```
입력 (1×28×28)
  → Conv(32) → BN → ReLU → Conv(32) → BN → ReLU → MaxPool(→14×14)
  → Conv(64) → BN → ReLU → Conv(64) → BN → ReLU → MaxPool(→7×7)
  → Flatten(3136) → Linear(512) → ReLU → Dropout → 출력(10개: 숫자 0~9)
```

In [ ]:
# ============================================================
# 🏗️ CNN 신경망 모델 정의하기
# ============================================================
# ★ 기존 MLP → CNN으로 업그레이드! ★
#
# [기존 MLP]  784 → 256 → 128 → 10
#   단점: 이미지를 1줄로 펼쳐서 공간 정보(위치, 모양)를 버림
#
# [CNN]  Conv → Conv → MaxPool → Conv → Conv → MaxPool → FC → 10
#   장점: 3×3 필터가 이미지를 훑으며 에지·곡선 등 패턴을 직접 학습!
#   → 정확도 ~97% → ~99% 향상
# ============================================================

class CNNClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        # ─── 특징 추출기 (Feature Extractor) ───
        # Conv2d(입력채널, 출력채널, 커널크기, padding)
        # padding=1 → 출력 크기를 입력과 동일하게 유지
        self.features = nn.Sequential(

            # ═══ 첫 번째 Conv 블록 ═══
            # 32개 필터가 3×3 창으로 이미지를 훑으며 기본 패턴(에지, 선) 추출
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # 1×28×28 → 32×28×28
            nn.BatchNorm2d(32),   # 배치 정규화: 학습 안정화
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),  # 32×28×28 → 32×28×28
            nn.BatchNorm2d(32),
            nn.ReLU(),

            # MaxPool2d: 2×2 영역에서 최댓값만 남김 → 크기 절반
            # 비유: 사진을 절반으로 축소해도 내용은 알아볼 수 있는 것처럼!
            nn.MaxPool2d(2),                               # 32×28×28 → 32×14×14
            nn.Dropout(0.25),

            # ═══ 두 번째 Conv 블록 ═══
            # 64개 필터로 더 복잡한 패턴(곡선, 교차점) 추출
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 32×14×14 → 64×14×14
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),  # 64×14×14 → 64×14×14
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 64×14×14 → 64×7×7
            nn.Dropout(0.25),
        )

        # ─── 분류기 (Classifier) ───
        # CNN이 뽑은 특징(64×7×7 = 3136개)을 이용해 최종 분류
        self.classifier = nn.Sequential(
            nn.Flatten(),              # 64×7×7 = 3136개로 펼치기
            nn.Linear(64 * 7 * 7, 512),   # 3136 → 512
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10)         # 512 → 10 (숫자 0~9)
        )

    def forward(self, x):
        x = self.features(x)     # Step A: 합성곱으로 패턴 추출
        x = self.classifier(x)   # Step B: 분류
        return x                  # Step C: 10개 숫자에 대한 점수 반환


# ─── 디바이스(GPU/CPU) 설정 ───
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 디바이스: {device}")

# ─── 모델 생성 ───
model = CNNClassifier().to(device)

print("\n📐 CNN 모델 구조:")
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\n🔢 총 학습 가능한 파라미터 수: {total_params:,}개")
print(f"   → 이 {total_params:,}개의 숫자를 조금씩 조절하는 게 '학습'이다!")
print(f"\n💡 기존 MLP 파라미터: 235,146개 → CNN: {total_params:,}개")

---
## ⚙️ Step 5. 손실 함수 & 옵티마이저 설정

- **손실 함수**: 모델이 얼마나 틀렸는지 점수 매기는 채점 기준
- **옵티마이저**: 그 점수를 보고 가중치를 어떻게 수정할지 결정하는 전략

In [ ]:
# ============================================================
# ⚙️ 학습에 필요한 도구 설정
# ============================================================

# CrossEntropyLoss: 분류 문제에서 가장 많이 쓰이는 손실 함수
criterion = nn.CrossEntropyLoss()

# Adam: 현재 가장 널리 쓰이는 옵티마이저
# lr=0.001: 학습률 (Learning Rate) = 한 번에 얼마나 크게 수정할지
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("✅ 손실 함수: CrossEntropyLoss (분류용)")
print("✅ 옵티마이저: Adam (lr=0.001)")
print("\n준비 완료! 이제 학습을 시작하겠습니다 🚀")

---
## 🏋️ Step 6. 모델 학습시키기

드디어 핵심! 아래 사이클을 수천 번 반복한다:

```
① 순전파: 이미지를 모델에 넣어서 예측값을 얻는다
② 손실 계산: 예측값과 정답을 비교해서 "얼마나 틀렸는지" 점수를 매긴다
③ 역전파: 거꾸로 추적하면서 "어떤 가중치가 틀림에 기여했는지" 계산한다
④ 업데이트: 문제가 된 가중치를 조금 수정한다
```

⏱️ GPU 기준 약 1~2분 소요됩니다.

In [ ]:
# ============================================================
# 🏋️ 학습 루프 (Training Loop)
# ============================================================

EPOCHS = 10
loss_history = []

print("🏋️ 학습을 시작합니다!\n")

for epoch in range(EPOCHS):

    # 학습 모드 켜기 (Dropout 활성화)
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # ★ STEP 1: 순전파
        outputs = model(images)

        # ★ STEP 2: 손실 계산
        loss = criterion(outputs, labels)

        # ★ STEP 3: 이전 그래디언트 초기화
        optimizer.zero_grad()

        # ★ STEP 4: 역전파
        loss.backward()

        # ★ STEP 5: 가중치 업데이트
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(f"  Epoch [{epoch+1:2d}/{EPOCHS}] | 평균 손실: {avg_loss:.4f}")

print("\n✅ 학습 완료!")
print("   손실이 점점 줄어들었다면 → 학습이 잘 된 것입니다! 🎉")

---
## 📉 Step 7. 학습 곡선 확인하기

에포크가 진행될수록 손실이 줄어드는지 그래프로 확인해보자!

In [ ]:
# ============================================================
# 📉 학습 곡선 (Loss Curve) 그리기
# ============================================================

plt.figure(figsize=(8, 4))

plt.plot(
    range(1, EPOCHS + 1),
    loss_history,
    marker='o',
    color='royalblue',
    linewidth=2
)

plt.title('학습 곡선 (Loss Curve)', fontsize=14)
plt.xlabel('Epoch (학습 반복 횟수)', fontsize=12)
plt.ylabel('Loss (손실 = 얼마나 틀렸나)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 그래프가 오른쪽 아래로 내려가면 학습이 잘 된 것!")
print(f"   첫 에포크 손실: {loss_history[0]:.4f} → 마지막 에포크 손실: {loss_history[-1]:.4f}")

---
## 📊 Step 8. 테스트 — 진짜 실력 확인하기

학습에 **한 번도 사용하지 않은** 테스트 데이터 10,000장으로 실력을 측정한다.

In [ ]:
# ============================================================
# 📊 테스트: 한 번도 본 적 없는 데이터로 실력 측정
# ============================================================

# 평가 모드 켜기 (Dropout 꺼짐)
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("=" * 40)
print(f"✅ 테스트 정확도: {accuracy:.2f}%")
print(f"   10,000장 중 {correct}장 맞춤!")
print("=" * 40)

if accuracy > 99:
    print("\n🎉 99%를 넘겼습니다! CNN의 위력을 느끼셨나요?")
elif accuracy > 97:
    print("\n🎉 97%를 넘겼습니다! 아주 잘 학습된 모델이에요!")
elif accuracy > 95:
    print("\n👍 95% 이상! 좋은 성능이에요!")
else:
    print("\n🤔 에포크를 더 늘리거나 모델 구조를 바꿔보세요!")

---
## 🔍 Step 9. 예측 결과 눈으로 확인하기

모델이 실제로 어떤 이미지를 보고 어떤 숫자라고 예측했는지 확인해보자!

- **파란색 제목** = 맞춘 것 ✅
- **빨간색 제목** = 틀린 것 ❌

In [ ]:
# ============================================================
# 🔍 예측 결과 시각화
# ============================================================

examples = iter(test_loader)
images, labels = next(examples)
images = images.to(device)
labels = labels.to(device)

model.eval()
with torch.no_grad():
    outputs = model(images)
    _, preds = torch.max(outputs, 1)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for i in range(10):
    row = i // 5
    col = i % 5
    ax = axes[row][col]

    ax.imshow(images[i].cpu().squeeze(), cmap='gray')

    true_label = labels[i].item()
    pred_label = preds[i].item()

    if true_label == pred_label:
        color = 'blue'
        result = '✅'
    else:
        color = 'red'
        result = '❌'

    ax.set_title(
        f"정답:{true_label} 예측:{pred_label} {result}",
        color=color,
        fontsize=11
    )
    ax.axis('off')

plt.suptitle('🔍 모델 예측 결과', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("파란색 = 맞춤 ✅ / 빨간색 = 틀림 ❌")

---
## 🧪 Step 10. (보너스) 내가 직접 숫자 그려서 테스트하기

Colab에서 직접 마우스로 숫자를 그려서 모델에 넣어보자!

In [ ]:
# ============================================================
# 🧪 직접 그린 숫자로 테스트하기
# ============================================================

from IPython.display import HTML, display
from google.colab import output
import numpy as np
from PIL import Image
import io, base64

def predict_drawing(image_data):
    image_data = image_data.split(',')[1]
    image_bytes = base64.b64decode(image_data)

    img = Image.open(io.BytesIO(image_bytes)).convert('L').resize((28, 28))
    img_array = np.array(img) / 255.0
    img_array = (img_array - 0.5) / 0.5

    img_tensor = torch.FloatTensor(img_array).unsqueeze(0).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        probabilities = torch.softmax(output, dim=1)
        predicted = torch.argmax(output, dim=1).item()
        confidence = probabilities[0][predicted].item() * 100

    print(f"\n🤖 모델 예측: {predicted} (확신도: {confidence:.1f}%)")
    print(f"\n각 숫자일 확률:")
    for i in range(10):
        bar = '█' * int(probabilities[0][i].item() * 30)
        print(f"  {i}: {probabilities[0][i].item()*100:5.1f}% {bar}")

output.register_callback('notebook.predict_drawing', predict_drawing)

canvas_html = """
<div style="text-align:center">
  <h3>✏️ 아래 검은 영역에 숫자를 그려보세요!</h3>
  <canvas id="drawCanvas" width="280" height="280"
    style="border:3px solid #4285f4; background:black; cursor:crosshair; border-radius:8px;">
  </canvas>
  <br><br>
  <button onclick="predictDrawing()"
    style="padding:10px 30px; font-size:16px; background:#4285f4; color:white;
           border:none; border-radius:6px; cursor:pointer; margin-right:10px;">
    🤖 예측하기
  </button>
  <button onclick="clearCanvas()"
    style="padding:10px 30px; font-size:16px; background:#ea4335; color:white;
           border:none; border-radius:6px; cursor:pointer;">
    🗑️ 지우기
  </button>
  <p style="color:gray; font-size:13px; margin-top:10px;">
    💡 팁: 캔버스 중앙에 크게 그려야 잘 인식됩니다!
  </p>
</div>
<script>
  var canvas = document.getElementById('drawCanvas');
  var ctx = canvas.getContext('2d');
  var drawing = false;
  ctx.strokeStyle = 'white';
  ctx.lineWidth = 18;
  ctx.lineCap = 'round';
  ctx.lineJoin = 'round';
  canvas.addEventListener('mousedown', function(e) {
    drawing = true; ctx.beginPath(); ctx.moveTo(e.offsetX, e.offsetY);
  });
  canvas.addEventListener('mousemove', function(e) {
    if (drawing) { ctx.lineTo(e.offsetX, e.offsetY); ctx.stroke(); }
  });
  canvas.addEventListener('mouseup', function() { drawing = false; });
  canvas.addEventListener('mouseout', function() { drawing = false; });
  canvas.addEventListener('touchstart', function(e) {
    e.preventDefault(); drawing = true;
    var touch = e.touches[0];
    var rect = canvas.getBoundingClientRect();
    ctx.beginPath(); ctx.moveTo(touch.clientX - rect.left, touch.clientY - rect.top);
  });
  canvas.addEventListener('touchmove', function(e) {
    e.preventDefault();
    if (drawing) {
      var touch = e.touches[0];
      var rect = canvas.getBoundingClientRect();
      ctx.lineTo(touch.clientX - rect.left, touch.clientY - rect.top);
      ctx.stroke();
    }
  });
  canvas.addEventListener('touchend', function() { drawing = false; });
  function clearCanvas() {
    ctx.fillStyle = 'black';
    ctx.fillRect(0, 0, canvas.width, canvas.height);
  }
  function predictDrawing() {
    var dataURL = canvas.toDataURL('image/png');
    google.colab.kernel.invokeFunction('notebook.predict_drawing', [dataURL], {});
  }
</script>
"""

display(HTML(canvas_html))
print("⬆️ 위 캔버스에 숫자를 그리고 [예측하기] 버튼을 눌러보세요!")

---

## 📝 실습 정리

### 오늘 배운 전체 흐름

```
📂 데이터 준비
  └→ MNIST 다운로드 & 전처리 (텐서 변환 + 정규화)

🏗️ 모델 설계 (CNN)
  └→ Conv블록(32채널) → Conv블록(64채널) → Linear(512) → 출력(10)

⚙️ 도구 설정
  └→ 손실 함수: CrossEntropyLoss
  └→ 옵티마이저: Adam (lr=0.001)

🏋️ 학습 (×10 에포크)
  └→ 순전파 → 손실 계산 → 역전파 → 가중치 업데이트

📊 평가
  └→ 테스트 데이터로 정확도 측정 (99%+ 도전!)
```

---

## 🔬 아래 추가 실험 셀들을 실행해보세요!

---
## 📈 실험 1. 학습률(Learning Rate) 비교

`lr=0.01`, `lr=0.001`, `lr=0.0001` 세 가지로 각각 **5 에포크**씩 학습해서
손실 곡선이 어떻게 달라지는지 비교해보자!

| 학습률 | 특징 |
|--------|------|
| `0.01` | 빠르게 학습하지만 불안정 (진동 가능) |
| `0.001` | 기본값 — 안정적이고 꾸준히 감소 ✅ |
| `0.0001` | 매우 안정적이지만 너무 느림 |

In [ ]:
# ============================================================
# 📈 실험 1. 학습률(lr) 비교
# ============================================================
# lr=0.01 / lr=0.001 / lr=0.0001 세 가지로 5 에포크씩 학습
# → 손실 곡선 변화를 비교!
# ============================================================

lr_list    = [0.01, 0.001, 0.0001]
lr_results = {}   # 각 lr의 에포크별 손실 저장

print("📈 학습률 비교 실험을 시작합니다!\n")

for lr in lr_list:
    print(f"  ── lr={lr} 학습 시작 ──")

    # 매번 새 모델 & 새 옵티마이저로 공정하게 비교
    model_lr = CNNClassifier().to(device)
    opt_lr   = optim.Adam(model_lr.parameters(), lr=lr)
    loss_tmp = []

    for epoch in range(5):
        model_lr.train()
        total = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            out  = model_lr(images)
            l    = criterion(out, labels)
            opt_lr.zero_grad(); l.backward(); opt_lr.step()
            total += l.item()
        avg = total / len(train_loader)
        loss_tmp.append(avg)
        print(f"    Epoch {epoch+1}/5 | 손실: {avg:.4f}")

    lr_results[lr] = loss_tmp
    print()

# ─── 그래프 비교 ───
plt.figure(figsize=(8, 4))
colors = ['tomato', 'royalblue', 'green']

for (lr, hist), color in zip(lr_results.items(), colors):
    plt.plot(range(1, 6), hist, marker='o', label=f'lr={lr}',
             color=color, linewidth=2)

plt.title('학습률 비교 — 손실 곡선 (5 에포크)', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 결과 해석:")
print("   lr=0.01   → 초반에 빠르게 내려가지만 진동(불안정)할 수 있음")
print("   lr=0.001  → 안정적이고 꾸준히 줄어듦 (기본값, 추천! ✅)")
print("   lr=0.0001 → 너무 천천히 학습 (에포크를 더 늘려야 함)")

---
## 📦 실험 2. 배치 사이즈(Batch Size) 비교

`batch=16`, `batch=64`, `batch=256` 으로 각각 **3 에포크**씩 학습해서
**속도**와 **손실**을 비교해보자!

| 배치 사이즈 | 특징 |
|------------|------|
| `16` | 업데이트 횟수 많음 → 느리지만 세밀한 학습 |
| `64` | 균형 잡힌 선택 ✅ |
| `256` | 빠르지만 업데이트 횟수 적음 |

In [ ]:
# ============================================================
# 📦 실험 2. 배치 사이즈(Batch Size) 비교
# ============================================================
# batch=16 / 64 / 256 으로 3 에포크씩 학습
# → 소요 시간 + 손실을 동시에 비교!
# ============================================================

import time

# AUTO-INJECTED: Korean font setup for matplotlib
import os as _os
import matplotlib.font_manager as _fm
import matplotlib.pyplot as _plt
if not any('NanumGothic' in f.name for f in _fm.fontManager.ttflist):
    for _font in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
                  '/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf']:
        if _os.path.exists(_font):
            _fm.fontManager.addfont(_font)
_plt.rcParams.update({'font.family': 'NanumGothic', 'axes.unicode_minus': False})
del _os, _fm, _plt
# END AUTO-INJECTED Korean font setup


batch_list    = [16, 64, 256]
batch_results = {}   # {batch_size: {'loss': [...], 'time': float}}

print("📦 배치 사이즈 비교 실험을 시작합니다!\n")

for bs in batch_list:
    print(f"  ── batch_size={bs} 학습 시작 ──")

    # 배치 사이즈에 맞는 DataLoader 새로 생성
    loader_bs = DataLoader(train_dataset, batch_size=bs, shuffle=True)

    model_bs = CNNClassifier().to(device)
    opt_bs   = optim.Adam(model_bs.parameters(), lr=0.001)
    loss_tmp = []

    start = time.time()
    for epoch in range(3):
        model_bs.train()
        total = 0
        for images, labels in loader_bs:
            images, labels = images.to(device), labels.to(device)
            out = model_bs(images)
            l   = criterion(out, labels)
            opt_bs.zero_grad(); l.backward(); opt_bs.step()
            total += l.item()
        avg = total / len(loader_bs)
        loss_tmp.append(avg)
        print(f"    Epoch {epoch+1}/3 | 손실: {avg:.4f}")

    elapsed = time.time() - start
    batch_results[bs] = {'loss': loss_tmp, 'time': elapsed}
    print(f"    ⏱️ 소요 시간: {elapsed:.1f}초\n")

# ─── 그래프 비교 ───
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
colors = ['tomato', 'royalblue', 'green']

for (bs, res), color in zip(batch_results.items(), colors):
    ax1.plot(range(1, 4), res['loss'], marker='o',
             label=f'batch={bs}', color=color, linewidth=2)

ax1.set_title('배치 사이즈별 손실 비교 (3 에포크)', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

bs_labels = [str(bs) for bs in batch_results.keys()]
times     = [res['time'] for res in batch_results.values()]
bars      = ax2.bar(bs_labels, times, color=colors, width=0.5)
ax2.set_title('배치 사이즈별 학습 시간 (3 에포크)', fontsize=13)
ax2.set_xlabel('Batch Size')
ax2.set_ylabel('시간 (초)')
for bar, t in zip(bars, times):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{t:.1f}s', ha='center', fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("💡 결과 해석:")
print("   batch=16  → 느리지만 안정적 (작은 배치 = 자주 업데이트)")
print("   batch=64  → 균형 잡힌 선택 (기본값, 추천! ✅)")
print("   batch=256 → 빠르지만 일반화 성능이 살짝 낮을 수 있음")

---
## 👗 실험 3. Fashion-MNIST 도전!

`datasets.MNIST` → `datasets.FashionMNIST` 로만 바꾼 것!  
똑같은 CNN 모델로 **옷 종류(티셔츠, 바지, 운동화 등)** 10가지를 분류해보자.

> 💡 MNIST(숫자)보다 정확도가 낮을 수 있어요 — 옷끼리는 생김새가 비슷하기 때문!

In [ ]:
# ============================================================
# 👗 실험 3. Fashion-MNIST 도전!
# ============================================================
# datasets.MNIST → datasets.FashionMNIST 로만 변경!
# 나머지 코드(모델, 학습 루프, 평가)는 완전히 동일
# ============================================================

fashion_class_names = [
    '티셔츠/탑', '바지', '풀오버', '드레스', '코트',
    '샌들',      '셔츠', '스니커즈', '가방',  '앵클부츠'
]

print("👗 Fashion-MNIST 데이터 준비 중...\n")

# ★ datasets.MNIST → datasets.FashionMNIST 로만 변경!
fashion_train = datasets.FashionMNIST(
    root='./data', train=True,  download=True, transform=transform
)
fashion_test = datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

fashion_train_loader = DataLoader(fashion_train, batch_size=64, shuffle=True)
fashion_test_loader  = DataLoader(fashion_test,  batch_size=64, shuffle=False)

print(f"✅ 학습 데이터: {len(fashion_train):,}장")
print(f"✅ 테스트 데이터: {len(fashion_test):,}장")

# ─── 데이터 미리보기 ───
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i in range(10):
    image, label = fashion_train[i]
    axes[i].imshow(image.squeeze(), cmap='gray')
    axes[i].set_title(fashion_class_names[label], fontsize=7)
    axes[i].axis('off')
plt.suptitle('👗 Fashion-MNIST 샘플 이미지', fontsize=12)
plt.tight_layout()
plt.show()

# ─── 학습 (5 에포크) ───
fashion_model     = CNNClassifier().to(device)
fashion_opt       = optim.Adam(fashion_model.parameters(), lr=0.001)
fashion_criterion = nn.CrossEntropyLoss()
fashion_loss_hist = []

print("\n🏋️ Fashion-MNIST CNN 학습 시작!\n")

for epoch in range(5):
    fashion_model.train()
    total = 0
    for images, labels in fashion_train_loader:
        images, labels = images.to(device), labels.to(device)
        out  = fashion_model(images)
        l    = fashion_criterion(out, labels)
        fashion_opt.zero_grad(); l.backward(); fashion_opt.step()
        total += l.item()
    avg = total / len(fashion_train_loader)
    fashion_loss_hist.append(avg)
    print(f"  Epoch [{epoch+1}/5] | 평균 손실: {avg:.4f}")

# ─── 테스트 정확도 ───
fashion_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in fashion_test_loader:
        images, labels = images.to(device), labels.to(device)
        _, predicted = torch.max(fashion_model(images), 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

fashion_acc = 100 * correct / total
print(f"\n{'='*40}")
print(f"✅ Fashion-MNIST 테스트 정확도: {fashion_acc:.2f}%")
print(f"   10,000장 중 {correct}장 맞춤!")
print(f"{'='*40}")
print("\n💡 MNIST(숫자)보다 정확도가 낮은 이유:")
print("   옷은 시각적으로 비슷한 것들이 많기 때문!")
print("   (예: 티셔츠 vs 셔츠, 풀오버 vs 코트)")

# ─── 예측 결과 시각화 ───
examples = iter(fashion_test_loader)
images, labels = next(examples)
images, labels = images.to(device), labels.to(device)

fashion_model.eval()
with torch.no_grad():
    _, preds = torch.max(fashion_model(images), 1)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i in range(10):
    row, col = i // 5, i % 5
    ax = axes[row][col]
    ax.imshow(images[i].cpu().squeeze(), cmap='gray')

    true_label = labels[i].item()
    pred_label = preds[i].item()
    color  = 'blue' if true_label == pred_label else 'red'
    result = '✅' if true_label == pred_label else '❌'

    ax.set_title(
        f"정답:{fashion_class_names[true_label]}\n예측:{fashion_class_names[pred_label]} {result}",
        color=color, fontsize=8
    )
    ax.axis('off')

plt.suptitle('👗 Fashion-MNIST CNN 예측 결과', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print("파란색 = 맞춤 ✅ / 빨간색 = 틀림 ❌")